# Lab 01 — RAG from Scratch

**Pairs with:** [Course 06 · RAG](https://psssnikhil.github.io/ai-engineering-handbook/build/module-09-rag-retrieval-augmented-generation/)

You will build a complete Retrieval-Augmented Generation pipeline with **no frameworks** — just chunking, TF-IDF retrieval, and the Anthropic SDK. By the end you'll understand exactly what LangChain and LlamaIndex abstract away, and you'll be able to implement this in a coding interview.

The pipeline:

```
documents → chunk → index (TF-IDF) → retrieve top-k → build grounded prompt → Claude → cited answer
```

**Prerequisites:** `pip install -r requirements.txt` and `ANTHROPIC_API_KEY` set in your environment. A full run costs a few cents.

In [ ]:
import anthropic
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
MODEL = "claude-opus-4-8"

## 1. A tiny document corpus

In production these would be your PDFs, wiki pages, or support tickets. Here we use a small in-memory knowledge base about a fictional company so retrieval quality is easy to eyeball.

In [ ]:
DOCUMENTS = {
    "refund-policy.md": """Acme Cloud Refund Policy. Customers on monthly plans may request a full refund
within 14 days of any charge. Annual plans are refundable within 30 days, prorated after that.
Refunds are processed to the original payment method within 5-7 business days.
Enterprise contracts follow the terms negotiated in the MSA and are not covered by this policy.""",

    "api-limits.md": """Acme Cloud API Rate Limits. Free tier: 60 requests per minute, 10,000 per day.
Pro tier: 600 requests per minute, 500,000 per day. Rate limit headers are returned on every
response: X-RateLimit-Remaining and X-RateLimit-Reset. Exceeding limits returns HTTP 429
with a Retry-After header. Burst allowances of 2x sustained rate apply for up to 30 seconds.""",

    "sso-setup.md": """Acme Cloud SSO Setup. Single sign-on via SAML 2.0 is available on Pro and Enterprise
tiers. To configure: open Admin > Security > SSO, upload your IdP metadata XML, and map the
email attribute. SCIM provisioning is Enterprise-only. Okta, Azure AD, and Google Workspace
are tested IdPs. Changes take up to 15 minutes to propagate.""",

    "data-retention.md": """Acme Cloud Data Retention. Deleted projects are soft-deleted and recoverable
for 30 days, after which they are purged permanently. Audit logs are retained for 1 year on
Pro and 3 years on Enterprise. Backups run nightly and are stored encrypted for 35 days.
Customers may request full data export at any time from Admin > Data.""",
}

print(f"{len(DOCUMENTS)} documents loaded")

## 2. Chunking

Real documents are too long to stuff into a prompt whole, and retrieval works better on focused passages. The classic baseline: **fixed-size chunks with overlap**. Overlap prevents a fact from being split across a chunk boundary and lost.

Our docs are small, so we use small chunks to make retrieval non-trivial. In production, 300–800 tokens per chunk is a common starting point — always tune against your own eval set.

In [ ]:
def chunk_text(text: str, chunk_size: int = 40, overlap: int = 10) -> list[str]:
    """Split text into word-based chunks with overlap."""
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size])
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(words):
            break
    return chunks

# Build the chunk store: each chunk remembers its source document
chunk_store = []  # list of {"source": ..., "text": ...}
for source, text in DOCUMENTS.items():
    for chunk in chunk_text(text):
        chunk_store.append({"source": source, "text": chunk})

print(f"{len(chunk_store)} chunks")
chunk_store[0]

## 3. Indexing and retrieval

Production systems use dense embeddings (e.g. voyage-3) in a vector database. The *mechanism* is identical with TF-IDF: turn every chunk into a vector, turn the query into a vector, rank by cosine similarity. We use TF-IDF here so the lab runs offline and you can inspect every number.

> Swapping `TfidfVectorizer` for an embedding API call is a one-function change — try it as the exercise at the end.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")
chunk_matrix = vectorizer.fit_transform(c["text"] for c in chunk_store)

def retrieve(query: str, k: int = 3) -> list[dict]:
    """Return the top-k chunks most similar to the query."""
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, chunk_matrix)[0]
    top_idx = np.argsort(scores)[::-1][:k]
    return [{**chunk_store[i], "score": float(scores[i])} for i in top_idx]

# Sanity check
for hit in retrieve("how long do I have to get my money back?"):
    print(f"{hit['score']:.3f}  {hit['source']}: {hit['text'][:70]}...")

Notice the retrieval nuance: the query says "money back" but the document says "refund" — lexical methods like TF-IDF partially miss this vocabulary gap (they match on weaker terms). This is exactly why dense embeddings exist: they match *meaning*, not words. Keep this example in your pocket for interviews.

## 4. Grounded generation

Now we stuff the retrieved chunks into the prompt and ask Claude to answer **only from that context**, with citations. Grounding instructions and a refusal path for out-of-scope questions are what separate RAG from vibes.

In [ ]:
SYSTEM = """You are a support assistant for Acme Cloud. Answer using ONLY the provided context.
Cite the source filename for every claim, like [refund-policy.md].
If the context does not contain the answer, say \"I don't have that information\" — never guess."""

def rag_answer(question: str, k: int = 3) -> str:
    hits = retrieve(question, k=k)
    context = "\n\n".join(f"<chunk source=\"{h['source']}\">\n{h['text']}\n</chunk>" for h in hits)
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=SYSTEM,
        messages=[{
            "role": "user",
            "content": f"<context>\n{context}\n</context>\n\nQuestion: {question}",
        }],
    )
    return next(b.text for b in response.content if b.type == "text")

print(rag_answer("How long do refunds take, and can I get one on an annual plan?"))

In [ ]:
# The refusal path: a question the corpus cannot answer
print(rag_answer("What is Acme Cloud's stock price?"))

## 5. A 10-line eval

Never ship RAG without measuring retrieval. The simplest useful metric is **hit rate**: for a set of questions with known source documents, how often does the right document appear in the top-k?

In [ ]:
GOLDEN_SET = [
    ("How do I set up Okta SSO?", "sso-setup.md"),
    ("What happens to deleted projects?", "data-retention.md"),
    ("What is the free tier rate limit?", "api-limits.md"),
    ("Can I get a refund after 20 days on an annual plan?", "refund-policy.md"),
    ("How long are audit logs kept?", "data-retention.md"),
]

hits = sum(
    expected in {h["source"] for h in retrieve(q, k=3)}
    for q, expected in GOLDEN_SET
)
print(f"Retrieval hit rate @3: {hits}/{len(GOLDEN_SET)} = {hits/len(GOLDEN_SET):.0%}")

## Exercises

1. **Swap in dense embeddings.** Replace TF-IDF with an embedding model (e.g. Voyage AI or sentence-transformers) and re-run the eval. Does the "money back" query improve?
2. **Break the chunking.** Set `overlap=0` and `chunk_size=15`. Which golden-set questions start failing, and why?
3. **Add answer evals.** Extend the golden set with expected answer facts and use Claude as a judge to score `rag_answer` outputs.
4. **Hybrid retrieval.** Combine TF-IDF scores with keyword-boosting for exact terms like error codes — a common production pattern.

**Next:** lab-02 builds a tool-using agent loop — the natural evolution from \"retrieve then answer\" to \"let the model decide when to retrieve.\""
